# PICON Tutorial

PICON (Persona Interrogation framework for Consistency evaluation) automatically interviews and evaluates LLM-based persona agents across three dimensions:

- **Internal Consistency**: Freedom from self-contradiction across answers
- **External Consistency**: Alignment of claims with real-world facts (via web search)
- **Retest Stability**: Consistency of answers when the same questions are repeated

## 1. Installation

In [ ]:
# Install from PyPI (simple)
#!pip install picon-eval

# Development install with all extras (Character.AI, Google GenAI, etc.)
#!pip install -e ".[all]"

In [ ]:
import picon
print(picon.__version__)

## 2. Environment Variables

Store API keys in a `.env` file or set them directly in the cell below.

In [ ]:
import os

# Option 1: Load from .env file automatically
from dotenv import load_dotenv, find_dotenv
dotenv_path = find_dotenv()
print("find_dotenv ->", repr(dotenv_path))          
# If env vars were already set in the process, load_dotenv() won't overwrite them by default.
# Use override=True to ensure values from .env replace placeholders (use carefully).
loaded = load_dotenv(dotenv_path, override=True) if dotenv_path else False
print("load_dotenv returned:", loaded)

# Option 2: Set directly
# os.environ["OPENAI_API_KEY"]  = "YOUR_OPENAI_KEY"          # Required for gpt-* models
# os.environ["GEMINI_API_KEY"]  = "YOUR_GEMINI_KEY"          # Required for gemini/* models
# os.environ["SERPER_API_KEY"]  = "YOUR_SERPER_KEY"          # Required for external verification
# os.environ["GOOGLE_GEOCODE"]  = "YOUR_GOOGLE_GEOCODE_KEY"  # Required for address validation

# Optional
# os.environ["ANTHROPIC_API_KEY"]   = "YOUR_ANTHROPIC_KEY"
# os.environ["GOOGLE_CLAIM_SEARCH"] = "YOUR_GOOGLE_API_KEY"
# os.environ["GOOGLE_CX_ID"]        = "YOUR_CUSTOM_SEARCH_ENGINE_ID"

print("All environment variables:"
      f"\nOPENAI_API_KEY: {os.getenv('OPENAI_API_KEY')[:8] + '...' if os.getenv('OPENAI_API_KEY') else 'Not Set'}"
      f"\nGEMINI_API_KEY: {os.getenv('GEMINI_API_KEY')[:8] + '...' if os.getenv('GEMINI_API_KEY') else 'Not Set'}"
      f"\nSERPER_API_KEY: {os.getenv('SERPER_API_KEY')[:8] + '...' if os.getenv('SERPER_API_KEY') else 'Not Set'}"
      f"\nGOOGLE_GEOCODE: {os.getenv('GOOGLE_GEOCODE')[:8] + '...' if os.getenv('GOOGLE_GEOCODE') else 'Not Set'}")    
print("Environment variables loaded successfully!")


## 3. Simple API: `picon.run()`

The simplest way to use PICON. Run interview + evaluation in just a few lines.

In [ ]:
import picon

result = picon.run(
    model="gpt-5-nano", #gemini/gemini-3-flash-preview
    persona="You are a 35-year-old software engineer living in San Francisco.",
    name="John",
    num_turns=50,
    num_sessions=2,
    do_eval=True,
    verbose=True,
)

print(result.eval_scores)
result.save("results/john.json")

## 4. Component-Based Usage

Compose your own pipeline by configuring each agent directly.

In [ ]:
from picon import Questioner, EntityExtractor, Evaluator, Interviewee
from picon import InterrogationSimulation

questioner   = Questioner(model="gpt-5")
extractor    = EntityExtractor(model="gpt-5.1")
evaluator    = Evaluator(model="gemini/gemini-2.5-flash")
interviewee  = Interviewee(
    model="gemini/gemini-3-flash-preview",
    persona="You are a 35-year-old software engineer living in San Francisco.",
    name="John",
)

sim = InterrogationSimulation(
    interviewee=interviewee,
    questioner=questioner,
    extractor=extractor,
    evaluator=evaluator,
    num_turns=50,
    num_sessions=2,
)
result = sim.run(do_eval=True, verbose=True)

print(result.eval_scores)
result.save("results/john_component.json")

## 5. Evaluate an External Persona Agent Endpoint

Evaluate any running OpenAI-compatible endpoint (`/v1/chat/completions`) directly.

In [ ]:
from picon import Interviewee, InterrogationSimulation

interviewee = Interviewee(api_base="http://localhost:8000/v1", name="MyAgent")
result = InterrogationSimulation(interviewee=interviewee, num_turns=50).run()

## 6. Self-hosted Model (vLLM)

```bash
# Start the vLLM server in a terminal first
vllm serve meta-llama/Llama-3-8B --port 8000
```

In [ ]:
from picon import Interviewee, InterrogationSimulation

interviewee = Interviewee(
    api_base="http://localhost:8000/v1",
    model="meta-llama/Llama-3-8B",
    persona="You are a 30-year-old teacher named Jane...",
    name="Jane",
)
result = InterrogationSimulation(interviewee=interviewee).run()

## 7. Separate Interview and Evaluation

In [ ]:
import picon

# Step 1: Interview only
interview_result = picon.run_interview(
    name="John",
    model="gemini/gemini-3-flash-preview",
    persona="You are a 35-year-old software engineer...",
    num_turns=50,
    num_sessions=2,
)

# Step 2: Evaluate (eval_factors: "internal", "external", "intra", "inter")
persona_stats = picon.run_evaluation(
    interview_result,
    eval_factors=["internal", "external"],
)
print(persona_stats)

## 8. Evaluate an Existing Interview File

In [ ]:
import picon

scores = picon.evaluate(
    "results/john.json",
    eval_factors=["internal", "external"],
)
print(scores)

## 9. Custom Wrapping Server (RAG, API calls, etc.)

Wrap any agent without an OpenAI-compatible endpoint using FastAPI.

In [ ]:
server_code = '''
import time
from fastapi import FastAPI, Request
import uvicorn

app = FastAPI()

def generate_response(messages: list) -> str:
    """Replace this with your own agent logic (RAG retrieval, API calls, etc.)."""
    user_message = messages[-1]["content"]
    return "This is my response."

@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    body = await request.json()
    content = generate_response(body.get("messages", []))
    return {
        "id": f"chatcmpl-{int(time.time())}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": "my-agent",
        "choices": [{"index": 0, "message": {"role": "assistant", "content": content}, "finish_reason": "stop"}],
        "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0},
    }

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8001)
'''

with open("server.py", "w") as f:
    f.write(server_code)

print("server.py created. Run `python server.py` in a terminal, then execute the cell below.")

In [ ]:
from picon import Interviewee, InterrogationSimulation

interviewee = Interviewee(api_base="http://localhost:8001/v1", name="MyCustomAgent")
result = InterrogationSimulation(interviewee=interviewee, num_turns=20).run(do_eval=True)
print(result.eval_scores)

## 10. Evaluation Metrics

| Metric | Description |
|--------|-------------|
| **Cooperativeness** | Fraction of turns with substantive, non-evasive responses |
| **Non-contradiction Rate** | Fraction of turns free of contradiction with prior responses |
| **Internal Consistency (IC)** | Harmonic mean of Cooperativeness and Non-contradiction Rate |
| **Coverage** | Fraction of turns containing at least one verifiable claim |
| **Non-refutation Rate** | Per-turn rate of claims not refuted by web evidence |
| **External Consistency (EC)** | Harmonic mean of Coverage and Non-refutation Rate |
| **Retest Consistency (Inter)** | Answer stability across sessions |
| **Retest Consistency (Intra)** | Answer stability within a session |

## 11. Paper Benchmark: 8 Persona Agent Types

End-to-end evaluation of the eight persona agent types used in the paper.

| # | Agent Type | Data Source | Notes |
|---|-----------|------------|-------|
| 1 | **LLM-Generated** | [Tianyi-Lab/Personas](https://huggingface.co/datasets/Tianyi-Lab/Personas) (HuggingFace) | Descriptive personas were selected |
| 2 | **Twin-2K-500** | [LLM-Digital-Twin/Twin-2K-500](https://huggingface.co/datasets/LLM-Digital-Twin/Twin-2K-500) (HuggingFace) | |
| 3 | **Nemotron** | [nvidia/Nemotron-Personas-*](https://huggingface.co/collections/nvidia/nemotron-personas-67ee07a5b2472f9f2e7d4977) (HuggingFace) | 7 regional datasets |
| 4 | **DeepPersona** | [arXiv:2511.07338](https://arxiv.org/abs/2511.07338) | |
| 5 | **Human Simulacra** | [arXiv:2402.18180](https://arxiv.org/abs/2402.18180) | RAG, Multi-agent System |
| 6 | **OpenCharacter** | [xywang1/OpenCharacter](https://huggingface.co/datasets/xywang1/OpenCharacter) (HuggingFace) | Requires fine-tuned vLLM server |
| 7 | **ConsistentLLM** | [arXiv:2511.00222](https://arxiv.org/abs/2511.00222) | Requires fine-tuned vLLM server |
| 8 | **Character.AI** | [character.ai](https://character.ai/) | Requires `CAI_TOKEN` |

### Common Configuration

All eight agents use the same PICON pipeline settings.

In [ ]:
# Shared pipeline config matching the paper's experimental setup
PICON_CONFIG = dict(
    questioner_model="gpt-5",
    extractor_model="gpt-5.1",
    web_search_model="gpt-5",
    evaluator_model="gemini/gemini-2.5-flash",
    num_turns=50,
    num_sessions=2,
    do_eval=True,
    verbose=True,
)

# Sampling settings (SAMPLE_N=0 runs all personas)
SAMPLE_N = 10
SEED = 42

---

### 11-1. LLM-Generated

Uses the `Tianyi-Lab/Personas` dataset. Choose one of four persona representation styles:
- `descriptive` (default): free-form narrative persona
- `objective`: objective table format
- `subjective`: subjective table format
- `meta`: meta persona

**Wrapping server**: `servers/llm_generated_server.py`

In [ ]:
import json
import subprocess
import tempfile
import time
import os
import requests
import picon
from datasets import load_dataset
from picon.env.interviewee_simulator.persona_prompt_builders import (
    build_llm_generated_prompt,
    extract_llm_generated_name,
)

LLM_GEN_PERSONA_TYPE    = "descriptive"  # descriptive | objective | subjective | meta
LLM_GEN_SIMULATOR_MODEL = "gemini/gemini-3-flash-preview"
BASE_PORT = 8100

dataset = load_dataset("Tianyi-Lab/Personas", split="train")
if SAMPLE_N > 0:
    dataset = dataset.shuffle(seed=SEED).select(range(min(SAMPLE_N, len(dataset))))

# Auto-detect available model prefix
available_prefixes = [c[:-len("_descriptive_persona")] for c in dataset.column_names if c.endswith("_descriptive_persona")]
prefix = next((p for p in ["Llama-3.1-70B-Instruct"] if p in available_prefixes), available_prefixes[0])

llm_gen_personas = []
for data in dataset:
    raw = {
        "descriptive_persona":      data.get(f"{prefix}_descriptive_persona", ""),
        "objective_table_persona":  data.get(f"{prefix}_objective_table_persona", ""),
        "subjective_table_persona": data.get(f"{prefix}_subjective_table_persona", ""),
        "meta_persona":             data.get("meta_persona", ""),
    }
    persona_prompt = build_llm_generated_prompt(json.dumps(raw), persona_type=LLM_GEN_PERSONA_TYPE)
    name = extract_llm_generated_name(raw["descriptive_persona"]) or f"LLM-Persona-{data.get('persona_number', '0')}"
    llm_gen_personas.append({"name": name, "prompt": persona_prompt})

print(f"Loaded {len(llm_gen_personas)} LLM-Generated personas (type={LLM_GEN_PERSONA_TYPE})")
print(f"First persona: {llm_gen_personas[0]['name']}")
print(f"Prompt preview: {llm_gen_personas[0]['prompt']}...")

In [ ]:
llm_gen_results = []

for i, persona in enumerate(llm_gen_personas):
    port = BASE_PORT + i + 1
    print(f"[{i+1}/{len(llm_gen_personas)}] {persona['name']} (port={port})")

    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, prefix="llmgen_") as f:
        f.write(persona["prompt"])
        tmpfile = f.name

    proc = subprocess.Popen([
        "python", "servers/llm_generated_server.py",
        "--port", str(port),
        "--model", LLM_GEN_SIMULATOR_MODEL,
        "--persona_file", tmpfile,
        "--name", persona["name"],
    ])

    for _ in range(30):
        try:
            if requests.get(f"http://localhost:{port}/health", timeout=1).ok:
                break
        except Exception:
            time.sleep(1)

    result = picon.run(
        api_base=f"http://localhost:{port}/v1",
        name=persona["name"],
        output_dir="data/results/llm_generated",
        **PICON_CONFIG,
    )
    llm_gen_results.append(result)

    proc.terminate()
    proc.wait()
    os.unlink(tmpfile)
    print(f"  Scores: {result.eval_scores}")

print(f"\nLLM-Generated done: {len(llm_gen_results)} personas")

---

### 11-2. Twin-2K-500

Loads personas from the `LLM-Digital-Twin/Twin-2K-500` digital twin dataset.

**Required packages**: `datasets`

**Wrapping server**: `servers/twin_2k_500_server.py`

In [ ]:
import json
import subprocess
import tempfile
import time
import os
import requests
import picon
from datasets import load_dataset
from picon.env.interviewee_simulator.persona_prompt_builders import build_twin_2k_500_prompt

TWIN_SIMULATOR_MODEL = "gemini/gemini-3-flash-preview"
BASE_PORT = 8100

dataset = load_dataset("LLM-Digital-Twin/Twin-2K-500", "full_persona", split="data")
if SAMPLE_N > 0:
    dataset = dataset.shuffle(seed=SEED).select(range(min(SAMPLE_N, len(dataset))))

twin_personas = []
for data in dataset:
    twin_personas.append({
        "name":   f"Twin-{data['pid']}",
        "prompt": build_twin_2k_500_prompt(json.dumps(data["persona_json"])),
    })

print(f"Loaded {len(twin_personas)} Twin-2K-500 personas")
print(f"First persona: {twin_personas[0]['name']}")
print(f"Prompt preview: {twin_personas[0]['prompt']}...")

In [ ]:
twin_results = []

for i, persona in enumerate(twin_personas):
    port = BASE_PORT + i + 1
    print(f"[{i+1}/{len(twin_personas)}] {persona['name']} (port={port})")

    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, prefix="twin_") as f:
        f.write(persona["prompt"])
        tmpfile = f.name

    proc = subprocess.Popen([
        "python", "servers/twin_2k_500_server.py",
        "--port", str(port),
        "--model", TWIN_SIMULATOR_MODEL,
        "--persona_file", tmpfile,
        "--name", persona["name"],
    ])

    for _ in range(30):
        try:
            if requests.get(f"http://localhost:{port}/health", timeout=1).ok:
                break
        except Exception:
            time.sleep(1)

    result = picon.run(
        api_base=f"http://localhost:{port}/v1",
        name=persona["name"],
        output_dir="data/results/twin_2k_500",
        **PICON_CONFIG,
    )
    twin_results.append(result)

    proc.terminate()
    proc.wait()
    os.unlink(tmpfile)
    print(f"  Scores: {result.eval_scores}")

print(f"\nTwin-2K-500 done: {len(twin_results)} personas")

---

### 11-3. Nemotron

Samples personas from NVIDIA's Nemotron-Personas datasets across 7 regions (USA, Korea, Singapore, France, India, Japan, Brazil).

**Required packages**: `datasets`

**Wrapping server**: `servers/nemotron_server.py`

In [ ]:
import json
import random
import subprocess
import time
import requests
import picon
from datasets import load_dataset
from picon.env.interviewee_simulator.persona_prompt_builders import build_nemotron_prompt

NEMOTRON_DATASETS = [
    ("nvidia/Nemotron-Personas-USA",       "usa",  "train"),
    ("nvidia/Nemotron-Personas-Korea",     "kor",  "train"),
    ("nvidia/Nemotron-Personas-Singapore", "sgp",  "train"),
    ("nvidia/Nemotron-Personas-France",    "fra",  "train"),
    ("nvidia/Nemotron-Personas-India",     "ind",  "en_IN"),
    ("nvidia/Nemotron-Personas-Japan",     "jpn",  "train"),
    ("nvidia/Nemotron-Personas-Brazil",    "bra",  "train"),
]
SIMULATOR_MODEL = "gemini/gemini-3-flash-preview"
BASE_PORT = 8100

# Allocate at least 1 per region, distribute remainder randomly
rng = random.Random(SEED)
n_groups = len(NEMOTRON_DATASETS)
quotas = [1] * n_groups
for _ in range(max(0, SAMPLE_N - n_groups)):
    quotas[rng.randrange(n_groups)] += 1

nemotron_personas = []
for (repo, region, split), quota in zip(NEMOTRON_DATASETS, quotas):
    ds = load_dataset(repo, split=split).shuffle(seed=SEED).select(range(min(quota, len(load_dataset(repo, split=split)))))
    for d in ds:
        uid = d.get("uuid", "unknown")
        nemotron_personas.append({
            "name":   f"Nemotron-{region.upper()}-{uid[:8]}",
            "prompt": build_nemotron_prompt(d),
        })

rng.shuffle(nemotron_personas)
print(f"Loaded {len(nemotron_personas)} Nemotron personas")
print(f"First persona: {nemotron_personas[0]['name']}")
print(f"Prompt preview: {nemotron_personas[0]['prompt']}...")

In [ ]:
import tempfile, os

nemotron_results = []

for i, persona in enumerate(nemotron_personas):
    port = BASE_PORT + i + 1
    print(f"[{i+1}/{len(nemotron_personas)}] {persona['name']} (port={port})")

    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, prefix="nemotron_") as f:
        f.write(persona["prompt"])
        tmpfile = f.name

    proc = subprocess.Popen([
        "python", "servers/nemotron_server.py",
        "--port", str(port),
        "--model", SIMULATOR_MODEL,
        "--persona_file", tmpfile,
        "--name", persona["name"],
    ])

    for _ in range(30):
        try:
            if requests.get(f"http://localhost:{port}/health", timeout=1).ok:
                break
        except Exception:
            time.sleep(1)

    result = picon.run(
        api_base=f"http://localhost:{port}/v1",
        name=persona["name"],
        output_dir="data/results/nemotron",
        **PICON_CONFIG,
    )
    nemotron_results.append(result)

    proc.terminate()
    proc.wait()
    os.unlink(tmpfile)
    print(f"  Scores: {result.eval_scores}")

print(f"\nNemotron done: {len(nemotron_results)} personas")

---

### 11-4. DeepPersona

Loads personas from local `.jsonl` profile files. The default path is `picon/env/personas/deeppersona`, which can be overridden by setting the `DATASET_DIR` environment variable.

Each line in a `.jsonl` file is a single profile, and the `profile_id` field is used as the persona identifier.

**Wrapping server**: `servers/deeppersona_server.py`

In [ ]:
import glob
import json
import os
import random
import subprocess
import tempfile
import time
import requests
from pprint import pprint
import picon
from picon.env.interviewee_simulator.persona_prompt_builders import build_deeppersona_prompt

DATASET_DIR = os.environ.get(
    "DATASET_DIR",
    "./picon/env/personas/deeppersona"
)
DEEPPERSONA_SIMULATOR_MODEL = "gemini/gemini-3-flash-preview"
BASE_PORT = 8100

if not os.path.isdir(DATASET_DIR):
    raise FileNotFoundError(f"DATASET_DIR does not exist: {DATASET_DIR}")

# Collect all (file, profile_id) pairs from .jsonl files then sample
files = sorted(glob.glob(os.path.join(DATASET_DIR, "*.jsonl")))
pairs = []
for f in files:
    with open(f) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            key = obj.get("profile_id", str(len(pairs)))
            pairs.append((f, key))

rng = random.Random(SEED)
if SAMPLE_N > 0:
    pairs = rng.sample(pairs, min(SAMPLE_N, len(pairs)))

deeppersona_personas = []
for filepath, profile_key in pairs:
    with open(filepath) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if obj.get("profile_id") == profile_key:
                deeppersona_personas.append({
                    "name":   profile_key,
                    "prompt": build_deeppersona_prompt(obj),
                })
                break

print(f"Loaded {len(deeppersona_personas)} DeepPersona personas from {DATASET_DIR}")
print(f"First persona: {deeppersona_personas[0]['name']}")
prompt = deeppersona_personas[0]['prompt']
split_idx = prompt.find('User profile:')
print(prompt[:split_idx + len('User profile:')])
print(prompt[split_idx + len('User profile:'):-1].strip())

In [ ]:
deeppersona_results = []

for i, persona in enumerate(deeppersona_personas):
    port = BASE_PORT + i + 1
    print(f"[{i+1}/{len(deeppersona_personas)}] {persona['name']} (port={port})")

    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, prefix="deeppersona_") as f:
        f.write(persona["prompt"])
        tmpfile = f.name

    proc = subprocess.Popen([
        "python", "servers/deeppersona_server.py",
        "--port", str(port),
        "--model", DEEPPERSONA_SIMULATOR_MODEL,
        "--persona_file", tmpfile,
        "--name", persona["name"],
    ])

    for _ in range(30):
        try:
            if requests.get(f"http://localhost:{port}/health", timeout=1).ok:
                break
        except Exception:
            time.sleep(1)

    result = picon.run(
        api_base=f"http://localhost:{port}/v1",
        name=persona["name"],
        output_dir="data/results/deeppersona",
        **PICON_CONFIG,
    )
    deeppersona_results.append(result)

    proc.terminate()
    proc.wait()
    os.unlink(tmpfile)
    print(f"  Scores: {result.eval_scores}")

print(f"\nDeepPersona done: {len(deeppersona_results)} personas")

---

### 11-5. Human Simulacra

Simulates 11 fixed characters using RAG. The wrapping server (`servers/human_simulacra_server.py`) is started automatically.

> RAG index loading takes longer, so the server health-check timeout is set to **60 seconds**.

**Required packages**: `langchain_google_genai`, `langchain_openai`

In [ ]:
import random
import subprocess
import time
import requests
import picon

ALL_CHARACTERS = [
    "Mary Jones",
    "Haley Collins",
    "Sara Ochoa",
    "James Jones",
    "Tami Clark",
    "Michael Miller",
    "Kevin Kelly",
    "Erica Walker",
    "Leslie Nichols",
    "Robert Scott",
    "Marsh Zhaleh",
]
HS_SIMULATOR_MODEL = "gemini/gemini-3-flash-preview"
BASE_PORT = 8100

rng = random.Random(SEED)
characters = rng.sample(ALL_CHARACTERS, min(SAMPLE_N, len(ALL_CHARACTERS))) if SAMPLE_N > 0 else ALL_CHARACTERS

print(f"Selected {len(characters)} Human Simulacra characters: {characters}")
print(f"First Persona: {characters[0]}")

In [ ]:
hs_results = []

for i, character_name in enumerate(characters):
    port = BASE_PORT + i + 1
    print(f"[{i+1}/{len(characters)}] {character_name} (port={port})")

    proc = subprocess.Popen([
        "python", "servers/human_simulacra_server.py",
        "--port", str(port),
        "--character_name", character_name,
        "--model", HS_SIMULATOR_MODEL,
    ])

    # Wait up to 60 seconds for RAG index to load
    for _ in range(60):
        try:
            if requests.get(f"http://localhost:{port}/health", timeout=1).ok:
                break
        except Exception:
            time.sleep(1)

    result = picon.run(
        api_base=f"http://localhost:{port}/v1",
        name=character_name,
        output_dir="data/results/human_simulacra",
        **PICON_CONFIG,
    )
    hs_results.append(result)

    proc.terminate()
    proc.wait()
    print(f"  Scores: {result.eval_scores}")

print(f"\nHuman Simulacra done: {len(hs_results)} characters")

---

### 11-6. OpenCharacter

Loads characters from the `xywang1/OpenCharacter` dataset and evaluates them using an OpenCharacter-SFT model served via vLLM.

**Prerequisite**: Start the vLLM server first.
```bash
vllm serve <your-opencharacter-model> --port 8000
```

**Wrapping server**: `servers/opencharacter_server.py`

In [ ]:
import re
import json
import subprocess
import tempfile
import time
import os
import requests
import picon
from datasets import load_dataset
from picon.env.interviewee_simulator.persona_prompt_builders import build_opencharacter_prompt

# Update to the address and model name of your running vLLM server
VLLM_BASE  = "http://localhost:8000/v1"
VLLM_MODEL = "anonymous/opencharacter-sft-llama-3-8b-instruct"
BASE_PORT  = 8100

dataset = load_dataset("xywang1/OpenCharacter", "Synthetic-Character", split="train")
if SAMPLE_N > 0:
    dataset = dataset.shuffle(seed=SEED).select(range(min(SAMPLE_N, len(dataset))))

openchar_personas = []
for data in dataset:
    name_match = re.match(r"Name:\s(.*)\n", data["character"])
    if not name_match:
        continue
    name = name_match.group(1).strip()
    prompt = build_opencharacter_prompt(data["persona"], data["character"])
    openchar_personas.append({"name": name, "prompt": prompt})

print(f"Loaded {len(openchar_personas)} OpenCharacter personas")
print(f"vLLM endpoint: {VLLM_BASE} / {VLLM_MODEL}")

In [ ]:
openchar_results = []

for i, persona in enumerate(openchar_personas):
    port = BASE_PORT + i + 1
    print(f"[{i+1}/{len(openchar_personas)}] {persona['name']} (port={port})")

    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, prefix="openchar_") as f:
        f.write(persona["prompt"])
        tmpfile = f.name

    proc = subprocess.Popen([
        "python", "servers/opencharacter_server.py",
        "--port", str(port),
        "--vllm_base", VLLM_BASE,
        "--vllm_model", VLLM_MODEL,
        "--persona_file", tmpfile,
        "--name", persona["name"],
    ])

    for _ in range(30):
        try:
            if requests.get(f"http://localhost:{port}/health", timeout=1).ok:
                break
        except Exception:
            time.sleep(1)

    result = picon.run(
        api_base=f"http://localhost:{port}/v1",
        name=persona["name"],
        output_dir="data/results/opencharacter",
        **PICON_CONFIG,
    )
    openchar_results.append(result)

    proc.terminate()
    proc.wait()
    os.unlink(tmpfile)
    print(f"  Scores: {result.eval_scores}")

print(f"\nOpenCharacter done: {len(openchar_results)} personas")

---

### 11-7. ConsistentLLM

Evaluates the ConsistentLLM fine-tuned model.

**Step 1 — Generate the persona file** (one-time setup):

Clone the ConsistentLLM repository and run the conversion script provided in this repo:
```bash
git clone https://github.com/abdulhaim/consistent-LLMs
python scripts/build_consistent_llm_personas.py \
    --personas_file consistent-LLMs/chatting/config_chatting_personas.json \
    --config_file   consistent-LLMs/chatting/config_chatting.json \
    --output_file   picon/env/personas/consistent_llm_personas.jsonl
```

**Step 2 — Start the vLLM server**:
```bash
vllm serve anonymous/consistent_llm_llama-8b-sft-ppo-prompt --port 8001
```

**Note**: No wrapping server needed — the persona JSON is passed directly to `Interviewee`.

In [ ]:
import json
import random
import picon
from picon import Interviewee, InterrogationSimulation, Questioner, EntityExtractor, Evaluator

PERSONAS_FILE   = "picon/env/personas/consistent_llm_personas.jsonl"
SIMULATOR_HOST  = "localhost"
SIMULATOR_PORT  = 8001
SIMULATOR_MODEL = "hosted_vllm/anonymous/consistent_llm_llama-8b-sft-ppo-prompt"

with open(PERSONAS_FILE) as f:
    all_personas = [json.loads(line) for line in f if line.strip()]

rng = random.Random(SEED)
if SAMPLE_N > 0:
    all_personas = rng.sample(all_personas, min(SAMPLE_N, len(all_personas)))

print(f"Loaded {len(all_personas)} ConsistentLLM personas")
print(f"Simulator: {SIMULATOR_MODEL} @ {SIMULATOR_HOST}:{SIMULATOR_PORT}")

In [ ]:
consistent_llm_results = []

for i, persona_data in enumerate(all_personas):
    name = persona_data.get("name", f"ConsistentLLM-{i}")
    print(f"[{i+1}/{len(all_personas)}] {name}")

    interviewee = Interviewee(
        api_base=f"http://{SIMULATOR_HOST}:{SIMULATOR_PORT}/v1",
        model=SIMULATOR_MODEL,
        persona=json.dumps(persona_data, ensure_ascii=False),
        name=name,
    )

    sim = InterrogationSimulation(
        interviewee=interviewee,
        questioner=Questioner(model=PICON_CONFIG["questioner_model"]),
        extractor=EntityExtractor(model=PICON_CONFIG["extractor_model"]),
        evaluator=Evaluator(model=PICON_CONFIG["evaluator_model"]),
        num_turns=PICON_CONFIG["num_turns"],
        num_sessions=PICON_CONFIG["num_sessions"],
        output_dir="data/results/consistent_llm",
    )
    result = sim.run(do_eval=PICON_CONFIG["do_eval"])
    consistent_llm_results.append(result)
    print(f"  Scores: {result.eval_scores}")

print(f"\nConsistentLLM done: {len(consistent_llm_results)} personas")

---

### 11-8. Character.AI

Evaluates real Character.AI characters. The character list is defined in `picon/env/personas/characterai.json`.

**Prerequisite**: A Character.AI session token (`CAI_TOKEN`) is required. Obtain it via [PyCharacterAI](https://github.com/Xtr4F/PyCharacterAI).

**Wrapping server**: `servers/characterai_server.py`

In [ ]:
import json
import random
import subprocess
import time
import os
import requests
import picon
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())  # Load environment variables from .env file if present

CAI_TOKEN     = os.environ.get("CAI_TOKEN", "")  # Recommended: set CAI_TOKEN in .env
PERSONAS_FILE = "picon/env/personas/characterai.json"
BASE_PORT     = 8100

if not CAI_TOKEN:
    raise ValueError("CAI_TOKEN is not set. Use os.environ['CAI_TOKEN'] = '...' or set it in .env.")

personas = json.load(open(PERSONAS_FILE))
rng = random.Random(SEED)
if SAMPLE_N > 0:
    personas = rng.sample(personas, min(SAMPLE_N, len(personas)))

print(f"Selected {len(personas)} Character.AI characters:")
for p in personas:
    print(f"  - {p['character_name']} (id={p['character_id']})")

In [ ]:
cai_results = []

for i, persona in enumerate(personas):
    port = BASE_PORT + i + 1
    name = persona["character_name"]
    cid  = persona["character_id"]
    print(f"[{i+1}/{len(personas)}] {name} (character_id={cid}, port={port})")

    proc = subprocess.Popen([
        "python", "servers/characterai_server.py",
        "--port", str(port),
        "--character_id", cid,
        "--user_id", CAI_TOKEN,
    ])

    for _ in range(30):
        try:
            if requests.get(f"http://localhost:{port}/", timeout=1).ok:
                break
        except Exception:
            time.sleep(1)

    result = picon.run(
        api_base=f"http://localhost:{port}/v1",
        name=name,
        output_dir="data/results/characterai",
        **PICON_CONFIG,
    )
    cai_results.append(result)

    proc.terminate()
    proc.wait()
    print(f"  Scores: {result.eval_scores}")

print(f"\nCharacter.AI done: {len(cai_results)} characters")

---

## 12. Aggregate Results & Comparison

Compare evaluation scores across all eight agent types in a single table.

In [ ]:
import statistics

def avg_scores(results: list, label: str) -> dict:
    """Compute per-metric averages from a list of PiconResult objects."""
    if not results:
        return {"agent": label}
    keys = [k for k in results[0].eval_scores.keys() if results[0].eval_scores[k] is not None]
    row = {"agent": label, "n": len(results)}
    for k in keys:
        vals = [r.eval_scores[k] for r in results if r.eval_scores.get(k) is not None]
        row[k] = round(statistics.mean(vals), 4) if vals else None
    return row

# Collect results from whichever agents were run
all_agent_results = {
    "Character.AI":   cai_results            if 'cai_results'            in dir() else [],
    "ConsistentLLM":  consistent_llm_results if 'consistent_llm_results' in dir() else [],
    "DeepPersona":    deeppersona_results    if 'deeppersona_results'    in dir() else [],
    "HumanSimulacra": hs_results             if 'hs_results'             in dir() else [],
    "LLM-Generated":  llm_gen_results        if 'llm_gen_results'        in dir() else [],
    "OpenCharacter":  openchar_results       if 'openchar_results'       in dir() else [],
    "Twin-2K-500":    twin_results           if 'twin_results'           in dir() else [],
    "Nemotron":       nemotron_results       if 'nemotron_results'       in dir() else [],
}

summary = [avg_scores(v, k) for k, v in all_agent_results.items() if v]

if summary:
    try:
        import pandas as pd
        df = pd.DataFrame(summary).set_index("agent")
        display(df)
    except ImportError:
        for row in summary:
            print(row)
else:
    print("No results available yet.")

In [ ]:
import json, os

os.makedirs("data/evaluation", exist_ok=True)
with open("data/evaluation/benchmark_summary.json", "w") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Benchmark summary saved to: data/evaluation/benchmark_summary.json")